# Text Embeddings with 🤗 `transformers` `pipeline`

This notebook uses the `feature-extraction` pipeline to turn text into embedding vectors, using a small, fast sentence-embedding model: `sentence-transformers/all-MiniLM-L6-v2` (~90 MB, 384-dim vectors).

Contents:
1. Basic embedding extraction
2. Mean-pooling raw token embeddings into a single sentence vector
3. Comparing sentences with cosine similarity
4. Semantic search over a small set of documents

In [ ]:
# If needed, uncomment to install dependencies
# %pip install -q transformers torch numpy --upgrade

## 1. Basic embedding extraction

The `feature-extraction` pipeline returns, for each input text, a list of token embeddings (one vector per token). We'll look at the raw shape first.

In [ ]:
from transformers import pipeline
import numpy as np

embedder = pipeline(
    "feature-extraction",
    model="sentence-transformers/all-MiniLM-L6-v2",
)

text = "Transformers make it easy to use state-of-the-art models."
raw_output = embedder(text)

raw_array = np.array(raw_output[0])  # [0] -> first (only) input in the batch
print("Shape (num_tokens, hidden_size):", raw_array.shape)

## 2. Mean-pooling into a single sentence embedding

For most use cases (similarity search, clustering, etc.) you want one fixed-size vector per sentence, not one per token. We do this by averaging (mean-pooling) the token embeddings.

In [ ]:
def embed_sentence(text: str) -> np.ndarray:
    token_embeddings = np.array(embedder(text)[0])   # (num_tokens, hidden_size)
    sentence_embedding = token_embeddings.mean(axis=0)  # (hidden_size,)
    return sentence_embedding

vec = embed_sentence(text)
print("Sentence embedding shape:", vec.shape)
print(vec[:10], "...")

## 3. Comparing sentences with cosine similarity

Cosine similarity tells us how semantically close two sentences are (1.0 = identical meaning, 0 = unrelated).

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    ("The cat sat on the mat.", "A cat was sitting on a mat."),
    ("The cat sat on the mat.", "The stock market crashed today."),
    ("I love programming in Python.", "Python is my favorite programming language."),
]

for a, b in pairs:
    sim = cosine_similarity(embed_sentence(a), embed_sentence(b))
    print(f"similarity={sim:.3f}\n  A: {a}\n  B: {b}\n")

## 4. Semantic search over a small document set

Embed a handful of "documents" once, then embed a query and rank documents by similarity — the core idea behind semantic/vector search.

In [ ]:
documents = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language for data science.",
    "The Great Wall of China stretches thousands of kilometers.",
    "Machine learning models can be trained on large datasets.",
    "Mount Everest is the tallest mountain in the world.",
]

doc_embeddings = [embed_sentence(doc) for doc in documents]

query = "Which language is good for AI and data analysis?"
query_embedding = embed_sentence(query)

scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

print(f"Query: {query}\n")
for doc, score in ranked:
    print(f"{score:.3f}  {doc}")

## Notes

- `sentence-transformers/all-MiniLM-L6-v2` is ~90 MB and produces 384-dimensional embeddings — a good default for lightweight semantic search/similarity tasks.
- If you have the `sentence-transformers` package installed, you can get pooled embeddings directly (no manual mean-pooling needed) via `pipeline("feature-extraction", model=..., framework="pt")` combined with `SentenceTransformer(...).encode(...)`, but the pure `transformers` approach above avoids the extra dependency.
- For production-scale semantic search, pair these embeddings with a vector index (e.g. FAISS) instead of the brute-force loop used here.